# Pré-processamento dos dados

Na EDA entendemos melhor a base e encontramos alguns pontos para tratar antes da modelagem. Agora vamos conferir os dados, retirar as repetições e separar treino e teste.

Também consideramos a análise da colega no fork: vamos manter o corte de qualidade em 7 e preservar os outliers.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline

# Localizando a pasta para abrir o notebook daqui ou da raiz do repositório
pastas = [Path.cwd(), Path.cwd() / "etapa_3", Path.cwd().parent / "etapa_3"]
pasta = next(p.resolve() for p in pastas if (p / "preprocessamento.py").exists())
resultados = pasta / "resultados"
resultados.mkdir(exist_ok=True)

df = pd.read_csv(pasta.parent / "data" / "WineQT.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


## Conferência da base
Partindo pelas mesmas verificações da EDA: tamanho da base, tipos das colunas e valores faltantes.

In [2]:
print("Linhas e colunas:", df.shape)
df.info()

Linhas e colunas: (1143, 13)
<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


In [3]:
nulos = df.isna().sum()
display(nulos.to_frame("nulos"))
nulos.rename("nulos").to_csv(resultados / "auditoria_nulos.csv")

,nulos
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


### Primeiros achados

A base tem 1.143 registros e não tem valores faltantes.
Considerando as 11 características físico-químicas como entradas do modelo. A nota e as classes de qualidade ficam de fora dessas entradas, pois são a resposta que queremos prever. O Id serve apenas como identificação e não se repete em nenhuma das linhas.

In [4]:
features = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide",
    "density", "pH", "sulphates", "alcohol"
]

# Se a base mudar, estas verificações ajudam a perceber problemas antes de seguir
assert set(features + ["quality", "Id"]).issubset(df.columns)
assert all(pd.api.types.is_numeric_dtype(df[coluna]) for coluna in features + ["quality", "Id"])
assert df[features + ["quality", "Id"]].notna().all().all()
assert np.isfinite(df[features + ["quality", "Id"]].to_numpy()).all()
assert df["quality"].between(0, 10).all()
assert (df["quality"] % 1 == 0).all()
assert df["Id"].is_unique
assert (df[features] >= 0).all().all()
assert (df[["density", "pH"]] > 0).all().all()
assert (df["free sulfur dioxide"] <= df["total sulfur dioxide"]).all()
print("Não encontramos problemas nessas verificações básicas.")

Não encontramos problemas nessas verificações básicas.


## Verificando as duplicadas

O Id é diferente em cada linha. Se ele entrar na comparação, podemos deixar de encontrar vinhos com os mesmos dados. Por isso, vamos comparar as características e a nota, sem considerar o Id.

In [5]:
colunas_comparacao = features + ["quality"]

# Repetições além da primeira ocorrência
repetidas = df.duplicated(subset=colunas_comparacao, keep="first")

# Todas as linhas que fazem parte de um grupo de registros iguais
envolvidas = df.duplicated(subset=colunas_comparacao, keep=False)

print("Repetições:", repetidas.sum())
print("Registros envolvidos:", envolvidas.sum())
print(f"Percentual de repetições: {repetidas.mean():.1%}")

Repetições: 125
Registros envolvidos: 239
Percentual de repetições: 10.9%


In [6]:
duplicadas = df.loc[envolvidas].sort_values(colunas_comparacao)
display(duplicadas.head(20))

# Guardando os registros para podermos consultar depois
duplicadas.to_csv(resultados / "grupos_duplicados.csv", index=False)
df.loc[repetidas].to_csv(resultados / "registros_removidos.csv", index=False)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
96,5.2,0.340,0.00,1.8,0.050,27.0,63.0,0.99160,3.68,0.79,14.0,6,142
98,5.2,0.340,0.00,1.8,0.050,27.0,63.0,0.99160,3.68,0.79,14.0,6,144
949,6.0,0.500,0.00,1.4,0.057,15.0,26.0,0.99448,3.36,0.45,9.5,5,1336
950,6.0,0.500,0.00,1.4,0.057,15.0,26.0,0.99448,3.36,0.45,9.5,5,1337
951,6.0,0.500,0.00,1.4,0.057,15.0,26.0,0.99448,3.36,0.45,9.5,5,1338
999,6.0,0.510,0.00,2.1,0.064,40.0,54.0,0.99500,3.54,0.93,10.7,6,1407
1001,6.0,0.510,0.00,2.1,0.064,40.0,54.0,0.99500,3.54,0.93,10.7,6,1409
517,6.4,0.570,0.02,1.8,0.067,4.0,11.0,0.99700,3.46,0.68,9.5,5,727
518,6.4,0.570,0.02,1.8,0.067,4.0,11.0,0.99700,3.46,0.68,9.5,5,728
603,6.4,0.640,0.21,1.8,0.081,14.0,31.0,0.99689,3.59,0.66,9.8,5,846


### Decisão sobre as duplicadas

Encontramos 125 repetições, envolvendo 239 linhas. Vamos manter a primeira ocorrência de cada combinação. Assim evitamos que um perfil igual apareça no treino e no teste, deixando o resultado do modelo melhor do que realmente é.

Não temos como afirmar que toda repetição é um erro de coleta. A base original será preservada, e os registros retirados ficam salvos para consulta.

In [7]:
df_limpo = df.drop_duplicates(subset=colunas_comparacao, keep="first").copy()

print("Antes:", len(df))
print("Depois:", len(df_limpo))

assert not df_limpo.duplicated(colunas_comparacao).any()
# Se o mesmo perfil tiver notas diferentes, precisamos investigar antes da divisão
assert not df_limpo.duplicated(features).any()

Antes: 1143
Depois: 1018


## Criando a variável alvo

Vamos manter a decisão da EDA: nota maior ou igual a 7 representa alta qualidade. Usaremos 1 para essa classe e 0 para baixa/média qualidade.

In [8]:
df_limpo["quality_class"] = (df_limpo["quality"] >= 7).astype(int)
display(df_limpo[["quality", "quality_class"]].head(10))
display(df_limpo["quality_class"].value_counts())

,quality,quality_class
0,5,0
1,5,0
2,5,0
3,6,0
5,5,0
6,5,0
7,7,1
8,7,1
9,5,0
10,5,0


quality_class
0    881
1    137
Name: count, dtype: int64

## Separando treino e teste

Vamos usar 80% dos dados para treino e 20% para teste. O `stratify` mantém uma proporção parecida das classes nos dois conjuntos. O `random_state=42` permite repetir a mesma divisão.

O teste fica reservado para avaliar os modelos depois. A escolha dos modelos e seus ajustes deve ser feita usando apenas o treino.

In [9]:
X = df_limpo[features]
y = df_limpo["quality_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Registros de treino:", len(X_train))
print("Registros de teste:", len(X_test))

# Conferindo se algum perfil aparece nos dois conjuntos
assert set(map(tuple, X_train.to_numpy())).isdisjoint(map(tuple, X_test.to_numpy()))
assert set(df_limpo.loc[X_train.index, "Id"]).isdisjoint(df_limpo.loc[X_test.index, "Id"])
assert len(X_train) + len(X_test) == len(df_limpo)

Registros de treino: 814
Registros de teste: 204


In [10]:
distribuicao = pd.DataFrame({
    "base_limpa": y.value_counts(),
    "treino": y_train.value_counts(),
    "teste": y_test.value_counts()
}).reindex([0, 1], fill_value=0)
distribuicao.index.name = "quality_class"

display(distribuicao)
display((distribuicao / distribuicao.sum() * 100).round(2))

,base_limpa,treino,teste
quality_class,,,
0,881,704,177
1,137,110,27


,base_limpa,treino,teste
quality_class,,,
0,86.54,86.49,86.76
1,13.46,13.51,13.24


### Achados da divisão

Ficamos com 814 registros para treino e 204 para teste. As classes continuam desbalanceadas, como vimos na EDA. Não vamos igualar as quantidades agora; na modelagem podemos comparar o uso de pesos de classe.

Como a EDA já usou a base inteira, esse teste não é totalmente novo em relação às análises anteriores. Precisamos mencionar essa limitação ao apresentar os resultados.

## Conferindo os outliers

A análise do fork indica que os extremos podem representar perfis reais de vinho. Vamos mantê-los, pois um valor fora do boxplot não significa necessariamente erro.

A tabela abaixo usa somente o treino para calcular os limites pelo IQR. Ela serve para acompanhar os valores extremos; não vamos retirar nem limitar esses registros.

In [11]:
outliers = []

for coluna in features:
    Q1 = X_train[coluna].quantile(0.25)
    Q3 = X_train[coluna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    mascara = (X_train[coluna] < limite_inferior) | (X_train[coluna] > limite_superior)

    outliers.append({
        "variavel": coluna,
        "q1_treino": Q1,
        "q3_treino": Q3,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "outliers_treino": int(mascara.sum())
    })

outliers = pd.DataFrame(outliers).set_index("variavel")
display(outliers.sort_values("outliers_treino", ascending=False))

,q1_treino,q3_treino,limite_inferior,limite_superior,outliers_treino
variavel,,,,,
residual sugar,1.900000,2.600000,0.850000,3.650000,66
chlorides,0.070000,0.091000,0.038500,0.122500,52
sulphates,0.550000,0.730000,0.280000,1.000000,35
fixed acidity,7.100000,9.175000,3.987500,12.287500,28
total sulfur dioxide,20.000000,61.000000,-41.500000,122.500000,28
density,0.995578,0.997827,0.992203,1.001202,25
pH,3.200000,3.400000,2.900000,3.700000,16
free sulfur dioxide,7.000000,21.000000,-14.000000,42.000000,9
volatile acidity,0.390000,0.645000,0.007500,1.027500,8


## Preparando a escala dos dados

As características têm escalas diferentes. Para os modelos que são sensíveis a isso, vamos usar o RobustScaler, que trabalha com a mediana e o intervalo entre os quartis.

Ele não remove os outliers nem transforma os dados em uma distribuição normal. Abaixo ajustamos a escala apenas no treino e aplicamos essa mesma transformação ao teste.

In [12]:
scaler = RobustScaler()

X_train_escala = scaler.fit_transform(X_train)
centro_treino = scaler.center_.copy()
X_test_escala = scaler.transform(X_test)

assert np.array_equal(centro_treino, scaler.center_)
assert np.isfinite(X_train_escala).all()
assert np.isfinite(X_test_escala).all()

display(pd.DataFrame(X_train_escala, columns=features).head())

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,-0.481928,-0.392157,-0.205882,0.142857,-0.380952,0.285714,-0.036585,-0.846667,0.425,-0.333333,0.411765
1,0.530120,-0.901961,0.205882,-0.214286,-0.761905,-0.071429,-0.256098,-0.668889,-0.225,1.222222,0.882353
2,-0.192771,-0.862745,0.470588,-0.285714,-1.285714,-0.571429,-0.329268,-0.162222,0.675,1.111111,1.000000
3,1.783133,-0.941176,0.617647,-0.714286,-1.190476,0.214286,-0.134146,0.086667,-0.775,0.277778,0.235294
4,-0.481928,-0.117647,0.676471,4.857143,-0.904762,0.857143,2.036585,0.326667,0.175,0.777778,0.176471


### Como vamos usar na modelagem

Na etapa 4, a escala deve ficar dentro de uma Pipeline junto com o classificador. Assim, durante a validação cruzada, ela será ajustada novamente em cada parte do treino.

Os arquivos de treino e teste serão salvos na escala original. A transformação acima é uma demonstração; não devemos usá-la como entrada de uma validação cruzada. Modelos de árvore podem trabalhar sem essa mudança de escala.

In [13]:
# Nova pipeline, ainda sem ajuste, para receber o classificador na etapa 4
preprocessador = Pipeline([
    ("escala", RobustScaler())
])
preprocessador

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('escala', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False


## Salvando os dados preparados

Vamos guardar os conjuntos separados e os registros usados na divisão. Os arquivos X e y de cada conjunto seguem a mesma ordem de linhas.

Por enquanto, manteremos as 11 características. Criar novas variáveis ou retirar algumas delas fica como possibilidade de comparação na etapa 4, usando apenas os dados de treino.

In [14]:
for nome, dados in [
    ("X_train", X_train), ("X_test", X_test),
    ("y_train", y_train), ("y_test", y_test)
]:
    dados.to_csv(resultados / f"{nome}.csv", index=False)

# Esta base inclui Id e nota apenas para consulta, não para alimentar o modelo
df_limpo.to_csv(resultados / "base_sem_duplicatas.csv", index=False)
distribuicao.to_csv(resultados / "distribuicao_classes.csv")
outliers.to_csv(resultados / "outliers_treino.csv")

divisao = pd.concat([
    df_limpo.loc[X_train.index, ["Id"]].assign(conjunto="treino"),
    df_limpo.loc[X_test.index, ["Id"]].assign(conjunto="teste")
])
divisao.to_csv(resultados / "divisao_por_id.csv", index=False)
print("Dados salvos na pasta resultados.")

Dados salvos na pasta resultados.


## Registrando as configurações
Também vamos salvar a identificação da base, as versões das bibliotecas e os parâmetros da divisão. Isso ajuda a conferir como os arquivos foram gerados.

In [15]:
import hashlib
import json
import sklearn

manifesto = {
    "fonte_relativa": "../data/WineQT.csv",
    "sha256": hashlib.sha256((pasta.parent / "data" / "WineQT.csv").read_bytes()).hexdigest(),
    "seed": 42,
    "proporcao_teste": 0.20,
    "alvo": "quality >= 7",
    "features": features,
    "registros_originais": len(df),
    "repeticoes_removidas": int(repetidas.sum()),
    "registros_em_grupos_duplicados": int(envolvidas.sum()),
    "registros_limpos": len(df_limpo),
    "registros_treino": len(X_train),
    "registros_teste": len(X_test),
    "outliers": "mantidos; limites IQR calculados no treino",
    "escala": "RobustScaler não ajustado; encaixar na pipeline do classificador",
    "versoes": {"pandas": pd.__version__, "numpy": np.__version__, "scikit-learn": sklearn.__version__}
}
(resultados / "manifesto.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Configurações salvas.")

Configurações salvas.


## Fechamento da etapa 3

Retiramos as repetições, mantivemos os outliers e separamos os dados em treino e teste. Também deixamos uma opção de escala pronta para usar com os modelos.

O próximo passo é treinar e comparar pelo menos dois classificadores. A escolha e os ajustes serão feitos no treino, deixando o teste para a avaliação final.